# Imvo project: all GPU work on Colab

**Use this in the browser** (colab.research.google.com): downloads do not work from the VS Code Colab extension.

1. **Runtime → Change runtime type → T4 GPU**.
2. Run all cells. Cell 2 asks for `colab/colab_bundle.zip` (make it with `python src/make_colab_bundle.py`).
3. The last cell downloads `embeddings_all.zip` (and `results_all.zip` if you ran the analysis cell). Unzip both into the project folder.

Total GPU time on a T4: ~40 min for LaBSE (4 spelling variants) + ~15 min for the second encoder.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
from google.colab import files
up = files.upload()  # choose colab_bundle.zip
!unzip -o -q colab_bundle.zip && ls src data/processed labels 2>/dev/null

Saving colab_bundle.zip to colab_bundle.zip
data/processed:
chunks.parquet	normaliser_candidates.parquet

src:
build_lexicon.py  embed.py		prepare_data.py  synthetic_check.py
classify.py	  make_colab_bundle.py	qualitative.py	 tool_metrics.py
common.py	  normalise.py		run_all.py
compare.py	  pairs.py		smoke_test.py


In [3]:
!pip -q install sentence-transformers pyarrow tabulate
# regenerate the modernised variants from the shipped candidate table (deterministic)
!for r in all asp prefix; do python src/normalise.py apply --rules $r; done

modern: 12.4% of isiXhosa tokens changed ({'asp': 289816, 'prefix_common': 634, 'prefix_proper': 169556}); 97.8% of chunks; thresholds 2,5 [provisional (no labelled words yet)]
modern_asp: 7.8% of isiXhosa tokens changed ({'asp': 289816}); 95.0% of chunks; thresholds 2,5 [provisional (no labelled words yet)]
modern_prefix: 4.6% of isiXhosa tokens changed ({'prefix_common': 633, 'prefix_proper': 169556}); 81.0% of chunks; thresholds 2,5 [provisional (no labelled words yet)]


## LaBSE: raw + three spelling variants
Modern variants reuse raw embeddings for unchanged chunks (all English text).

In [4]:
!python src/embed.py --variant raw --batch 128 --shard 8192
!python src/embed.py --variant modern --chunks data/processed/chunks_modern.parquet --reuse raw --batch 128 --shard 8192
!python src/embed.py --variant modern_asp --chunks data/processed/chunks_modern_asp.parquet --reuse raw --batch 128 --shard 8192
!python src/embed.py --variant modern_prefix --chunks data/processed/chunks_modern_prefix.parquet --reuse raw --batch 128 --shard 8192

modules.json: 100% 461/461 [00:00<00:00, 1.83MB/s]
config_sentence_transformers.json: 100% 122/122 [00:00<00:00, 665kB/s]
README.md: 100% 2.02k/2.02k [00:00<00:00, 7.15MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 250kB/s]
config.json: 100% 804/804 [00:00<00:00, 4.15MB/s]

model.safetensors: downloading bytes:   8% 143M/1.88G [00:01<00:08, 210MB/s, 9.37MB/s  ] 
model.safetensors: downloading bytes:  14% 261M/1.88G [00:01<00:05, 301MB/s, 20.7MB/s  ]
model.safetensors: downloading bytes:  16% 307M/1.88G [00:01<00:05, 268MB/s, 27.5MB/s  ]
model.safetensors: downloading bytes:  35% 652M/1.88G [00:02<00:04, 292MB/s, 56.2MB/s  ]
model.safetensors: downloading bytes:  39% 730M/1.88G [00:03<00:04, 277MB/s, 60.2MB/s  ]
model.safetensors: downloading bytes:  52% 970M/1.88G [00:03<00:03, 294MB/s, 77.1MB/s  ]
model.safetensors: downloading bytes:  57% 1.06G/1.88G [00:04<00:03, 254MB/s, 82.5MB/s  ]
model.safetensors: downloading bytes:  62% 1.18G/1.88G [00:04<00:02, 306MB/s, 89.9MB/

## Second encoder (robustness): multilingual-e5-base, raw + modern

In [5]:
E5 = 'intfloat/multilingual-e5-base'
!python src/embed.py --variant raw --model {E5} --prefix 'query: ' --batch 128 --shard 8192
!python src/embed.py --variant modern --chunks data/processed/chunks_modern.parquet --model {E5} --prefix 'query: ' --reuse raw --batch 128 --shard 8192

modules.json: 100% 387/387 [00:00<00:00, 1.79MB/s]
README.md: 100% 179k/179k [00:00<00:00, 111MB/s]
sentence_bert_config.json: 100% 57.0/57.0 [00:00<00:00, 242kB/s]
config.json: 100% 694/694 [00:00<00:00, 3.47MB/s]

model.safetensors: downloading bytes:  27% 295M/1.11G [00:01<00:03, 255MB/s, 26.1MB/s  ]
model.safetensors: downloading bytes:  30% 338M/1.11G [00:01<00:03, 202MB/s, 29.2MB/s  ]
model.safetensors: downloading bytes:  41% 460M/1.11G [00:02<00:02, 222MB/s, 38.8MB/s  ]
model.safetensors: downloading bytes:  67% 741M/1.11G [00:03<00:01, 228MB/s, 59.7MB/s  ]
model.safetensors: reconstructing file:  48% 536M/1.11G [00:04<00:06, 93.9MB/s, 36.4MB/s  ]
model.safetensors: downloading bytes: 100% 741M/741M [00:15<00:00, 48.8MB/s, 60.6MB/s  ]
model.safetensors: reconstructing file: 100% 1.11G/1.11G [00:15<00:00, 73.2MB/s, 62.2MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 5550.64it/s]
tokenizer_config.json: 100% 418/418 [00:00<00:00, 2.13MB/s]

sentencepiece.bpe.model: downloading

In [6]:
import numpy as np, glob
for f in sorted(glob.glob('data/embeddings/*.npy')):
    e = np.load(f); n = np.linalg.norm(e.astype('float32'), axis=1)
    print(f, e.shape, 'norms', n.min().round(3), n.max().round(3))

data/embeddings/modern_LaBSE.npy (68392, 768) norms 1.0 1.0
data/embeddings/modern_asp_LaBSE.npy (68392, 768) norms 1.0 1.0
data/embeddings/modern_multilingual-e5-base.npy (68392, 768) norms 1.0 1.0
data/embeddings/modern_prefix_LaBSE.npy (68392, 768) norms 1.0 1.0
data/embeddings/raw_LaBSE.npy (68392, 768) norms 1.0 1.0
data/embeddings/raw_multilingual-e5-base.npy (68392, 768) norms 1.0 1.0


## Optional: run the analysis here too
Needs `labels/pairs_labelled.csv` in the bundle, or a provisional threshold.

In [7]:
THRESHOLD = 0.55   # provisional; ignored if labels/pairs_labelled.csv was bundled
!python src/run_all.py analyse --threshold {THRESHOLD}
!python src/run_all.py analyse --model-tag multilingual-e5-base --threshold {THRESHOLD}


$ python src/pairs.py candidates --variant raw --model-tag LaBSE
8,066 mutual-NN candidates from 511 issues (splits embedded: ['test', 'train', 'val'])
        count   mean    std    min    25%    50%    75%    max
split                                                         
test   1243.0  0.509  0.080  0.328  0.451  0.493  0.550  0.792
train  5618.0  0.509  0.079  0.272  0.454  0.494  0.548  0.875
val    1205.0  0.511  0.077  0.297  0.458  0.492  0.549  0.821

$ python src/pairs.py candidates --variant modern --model-tag LaBSE
8,188 mutual-NN candidates from 511 issues (splits embedded: ['test', 'train', 'val'])
        count   mean    std    min    25%    50%    75%    max
split                                                         
test   1240.0  0.515  0.082  0.322  0.457  0.500  0.556  0.821
train  5709.0  0.514  0.080  0.272  0.457  0.498  0.554  0.873
val    1239.0  0.515  0.079  0.337  0.459  0.497  0.558  0.831

$ python src/pairs.py candidates --variant modern_asp --mode

In [8]:
!cd data/embeddings && zip -q -0 ../../embeddings_all.zip *.npy *_ids.txt
!zip -q -r results_all.zip results 2>/dev/null; ls -la *.zip
files.download('embeddings_all.zip')
import os
if os.path.exists('results_all.zip'): files.download('results_all.zip')

-rw-r--r-- 1 root root  26667022 Oct 18 14:04 colab_bundle.zip
-rw-r--r-- 1 root root 636595192 Oct 18 15:05 embeddings_all.zip
-rw-r--r-- 1 root root  23893918 Oct 18 15:05 results_all.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>